# Retail EDA - Day 4
This notebook loads all Day 2 and Day 3 CSV outputs and visualizes key business KPIs.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
ROOT = Path.cwd()
DAY2_DIR = ROOT / 'day2_results'
DAY3_DIR = ROOT / 'day3_results'

In [ ]:
day2_data = {f.stem: pd.read_csv(f) for f in sorted(DAY2_DIR.glob('*.csv'))}
day3_data = {f.stem: pd.read_csv(f) for f in sorted(DAY3_DIR.glob('*.csv'))}

print('Day 2 files loaded:', len(day2_data))
print('Day 3 files loaded:', len(day3_data))
print('Day 2 keys:', list(day2_data.keys()))
print('Day 3 keys:', list(day3_data.keys()))

In [ ]:
monthly_rev = day3_data['05_monthly_running_revenue'].copy()
monthly_rev['revenue_month'] = pd.to_datetime(monthly_rev['revenue_month'])

region_store = day2_data['01_revenue_by_region_store_type'].copy()
region_rev = region_store.groupby('region', as_index=False)['revenue'].sum()

rfm_summary = day3_data['01b_rfm_segment_summary'].copy()
cohort = day3_data['02_cohort_retention'].copy()
cohort['cohort_month'] = pd.to_datetime(cohort['cohort_month'])

sales = pd.read_csv(ROOT / 'bm_sales.csv', parse_dates=['date'])
skus = pd.read_csv(ROOT / 'bm_skus.csv')
top_products = (sales.merge(skus[['sku_id', 'sku_name']], on='sku_id', how='left')
                 .groupby(['sku_id', 'sku_name'], as_index=False)['total_value'].sum()
                 .sort_values('total_value', ascending=False)
                 .head(10))

## 1) Line Chart: Monthly Revenue Trend

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_rev['revenue_month'], monthly_rev['monthly_revenue'], color='#1f77b4', linewidth=2)
ax.set_title('Monthly Revenue Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 2) Bar Chart: Revenue by Region

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(region_rev['region'], region_rev['revenue'], color=['#2a9d8f', '#e9c46a', '#e76f51'])
ax.set_title('Revenue by Region')
ax.set_xlabel('Region')
ax.set_ylabel('Revenue')
plt.tight_layout()
plt.show()

## 3) Bar Chart: Top 10 Products by Revenue

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_df = top_products.sort_values('total_value', ascending=True)
ax.barh(plot_df['sku_name'], plot_df['total_value'], color='#264653')
ax.set_title('Top 10 Products by Revenue')
ax.set_xlabel('Revenue')
ax.set_ylabel('Product')
plt.tight_layout()
plt.show()

## 4) Pie Chart: RFM Segment Mix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.pie(
    rfm_summary['customers'],
    labels=rfm_summary['rfm_segment'],
    autopct='%1.1f%%',
    startangle=90
)
ax.set_title('RFM Customer Segments')
plt.tight_layout()
plt.show()

## 5) Heatmap: Cohort Retention (%)

In [ ]:
heat = cohort.pivot_table(index='cohort_month', columns='months_since_signup', values='retention_pct', aggfunc='mean')

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(heat.values, aspect='auto', cmap='YlOrRd', interpolation='nearest')
ax.set_title('Cohort Retention Heatmap')
ax.set_xlabel('Months Since Signup')
ax.set_ylabel('Cohort Month')
ax.set_xticks(range(len(heat.columns)))
ax.set_xticklabels(heat.columns, rotation=45)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels([d.strftime('%Y-%m') for d in heat.index])
plt.colorbar(im, ax=ax, label='Retention %')
plt.tight_layout()
plt.show()